# 03 — Functions & Methods

Notebook 02 gave you values. This notebook gives you the verbs that act on them.

Scala has two related-but-distinct ideas: **methods** (defined with `def`, attached to an enclosing class, object, or scope) and **functions** (first-class values you can pass around like integers). Understanding the small gap between them — and how Scala bridges it for you — is the single most important skill for reading idiomatic code, especially Spark code that is built end-to-end on passing functions to operators like `map` and `filter`.

## Defining a method with `def`

The keyword `def` introduces a method. The shape is:

```
  def name(param1: Type1, param2: Type2): ReturnType =
    body-expression
```

The body is a single expression. Its value is what the method returns — no `return` keyword needed.

In [ ]:
def square(x: Int): Int = x * x

def greet(name: String): String =
  s"hello, $name"

square(7)        // 49
greet("ganesh")  // "hello, ganesh"

A multi-line body becomes a block expression — same rules as notebook 02. The last expression in the block is the method's return value.

In [ ]:
def hypotenuse(a: Double, b: Double): Double =
  val a2 = a * a
  val b2 = b * b
  math.sqrt(a2 + b2)

## Parameter types are mandatory

Scala will infer the type of a `val`. It will **not** infer the types of method parameters. Every parameter must carry a type annotation. If you forget one, the compiler stops you.

Why this rule? A method is a contract — callers need to know what they can pass in. Inference at the call site would be fragile and hard to read.

## Default and named arguments

Parameters can have **default values**, used when the caller omits them. Callers can also pass arguments by **name** to skip positional ordering.

In [ ]:
def greet(name: String, greeting: String = "hello"): String =
  s"$greeting, $name"

greet("ganesh")                       // "hello, ganesh"
greet("ganesh", "hi")                 // "hi, ganesh"
greet(name = "ganesh", greeting = "namaste")
greet(greeting = "hi", name = "ganesh")   // order doesn't matter when named

Defaults let you grow an API without breaking callers. Named arguments make calls self-documenting — especially valuable when several parameters share the same type (e.g., three `Int`s in a row).

## Return types: write them at boundaries

Scala can infer a method's return type from its body. But the inferred type leaks the implementation. The same guidance as in notebook 02 applies, just sharper:

- **Public methods** — write the return type. It documents the contract; a body change that subtly changes the type will fail loudly.
- **Local / private helpers** — let inference do its job.
- **Recursive methods** — the return type is **required**. The compiler cannot infer the type of something defined in terms of itself.

In [ ]:
def factorial(n: Int): Int =        // return type required for recursion
  if n <= 1 then 1
  else n * factorial(n - 1)

## `Unit`-returning methods

A method that returns `Unit` is called for its **side effect** — printing, mutating, writing to a file. It returns the value `()`, which carries no information.

In [ ]:
def announce(name: String): Unit =
  println(s"hello, $name")
  // no return value worth caring about

There is a deprecated short form where you omit the `=` sign — `def foo(...) { ... }`. Avoid it in Scala 3. The explicit `: Unit =` form is clearer about intent: this method is here for its side effect.

## Functions are values

Methods live attached to a scope. **Function values** — also called lambdas or anonymous functions — are objects you can name, store, and pass around. The syntax is `(params) => body`.

In [ ]:
val double = (x: Int) => x * 2
val add = (a: Int, b: Int) => a + b

double(7)       // 14
add(3, 4)       // 7

The type of a function value is written with `=>` between the parameter list and the result type. The two values above have these types:

```
  double : Int => Int
  add    : (Int, Int) => Int
```

That arrow notation appears everywhere in Scala. Read it as *takes ... returns ...*. `Int => Int` is *takes an Int, returns an Int*. `(String, Int) => Boolean` is *takes a String and an Int, returns a Boolean*.

## Higher-order functions

A **higher-order function** is one that takes a function as an argument, returns a function as a result, or both. This is the bread and butter of FP-style Scala and of Spark's API.

In [ ]:
def applyTwice(f: Int => Int, x: Int): Int = f(f(x))

applyTwice(double, 5)         // double(double(5)) = 20
applyTwice((x: Int) => x + 1, 10)   // 12
applyTwice(_ + 1, 10)               // 12, using the underscore shorthand

The last form, `_ + 1`, is shorthand for `x => x + 1`. The underscore stands for a parameter that is used exactly once, in order. It is idiomatic for short lambdas. Use the full `x =>` form when the body is non-trivial.

Functions can also be **returned** from other functions:

In [ ]:
def adder(n: Int): Int => Int =
  (x: Int) => x + n

val addFive = adder(5)
addFive(10)     // 15
addFive(20)     // 25

`adder(5)` returns a *new function* with the value `5` baked in. That captured `5` is called the function's **closure** — see the closures section below.

## Methods vs functions (the gap)

Here is the precise distinction:

```text
  method:   def square(x: Int): Int = x * x
            -- attached to a scope, not a value by itself

  function: val square = (x: Int) => x * x
            -- a value of type Int => Int, can be passed around
```

Methods are not first-class values. Functions are. So how do you pass a method where a function value is expected?

## Eta expansion

Scala's answer is **eta expansion**: the compiler automatically converts a method into a function value when the context expects one.

In [ ]:
def square(x: Int): Int = x * x      // a method

val nums = List(1, 2, 3, 4)
nums.map(square)                     // method passed where a function is expected
// List(1, 4, 9, 16)

val squareFn: Int => Int = square    // explicit eta expansion
squareFn(7)                          // 49

In both lines, the compiler quietly wrote `(x: Int) => square(x)` for you. In Scala 3 this happens **automatically** in any context that expects a function type. If you ever need to force it explicitly, append a trailing underscore: `square _`. You will rarely need to.

Practical takeaway: write methods with `def` (most code), define them once, and let the compiler hand them to `map`/`filter`/`flatMap` as function values when needed.

## Multiple parameter lists

A Scala method can have more than one parameter list. The body can use parameters from all of them.

In [ ]:
def add(a: Int)(b: Int): Int = a + b

add(3)(4)            // 7

val addThree = add(3)   // partially applied: gives back Int => Int
addThree(10)            // 13

Two payoffs from this shape:

- **Partial application** — call the method with the first list only and get a function back, ready to be called with the rest. We just did exactly that above.
- **Type inference flows left to right.** A type the compiler infers from the first list is known by the time it reads the second list. Notebook 11 leans on this hard when we meet `given` and `using`.

Spark itself uses this shape for some operators where the first list specifies what to do and a later list specifies how to do it (e.g., partial application for builder-style APIs).

## Closures

A **closure** is a function value that captures references to variables in the surrounding scope at the moment it was created.

In [ ]:
def adder(n: Int): Int => Int =
  (x: Int) => x + n      // captures `n` from the enclosing scope

val plus10 = adder(10)
val plus100 = adder(100)

plus10(5)     // 15
plus100(5)    // 105

Each call to `adder` returns a *different* function value, because each one closes over a different `n`. This is the cleanest way to package a small piece of state with the behaviour that uses it.

Closures matter enormously in Spark. When you write `rdd.map(x => x + offset)`, Spark serialises that closure — including the captured `offset` — and ships it to every executor in the cluster. If `offset` were a mutable variable that lives only on the driver, the executors would see a stale copy. Notebook 15 will revisit closures from the serialisation angle.

## Putting it together

Here is a small example that uses every piece from this notebook: a method, a function value, a higher-order method that takes a function, and a closure.

In [ ]:
// 1) a method
def isAbove(threshold: Int)(value: Int): Boolean = value > threshold

// 2) partial application produces a function value with `threshold` baked in
val isAboveTen: Int => Boolean = isAbove(10)

// 3) higher-order method `filter` takes a function value
val nums = List(3, 8, 12, 1, 15, 7)
val big = nums.filter(isAboveTen)
// big: List[Int] = List(12, 15)

// 4) same idea with an inline lambda — what you'll see 90% of the time
val bigger = nums.filter(_ > 10)
// bigger: List[Int] = List(12, 15)

Read the closing two lines side by side. The first builds a reusable function value and uses it. The second writes the predicate inline. Both compile to the same bytecode in modern Scala. Style choice: extract when the predicate is named, complex, or used more than once; inline when it is small and used once.

## What's next

You can now write methods, define functions as values, pass them to higher-order operators, and capture state in closures. Notebook 04 introduces **collections** — `List`, `Vector`, `Map`, `Set`, `Array` — the data shapes that all the higher-order functions will operate on. After that, notebook 05 ties it together: `map`, `filter`, `fold`, and `for ... yield` over real data.